# CIS 5450 Project: Difficulty Topics
**Team Members:** Sejal Kotian, [teammate 2], [teammate 3], [teammate 4]

> This notebook documents how we implemented the three difficulty topics in our project.  
> All three concepts are implemented in **`final_notebook.ipynb`**.  
> Use the link button in the top right when you select a cell to get a **hyperlink**.


---

## Topic 1: **Feature Engineering — Time-Aware Feature Engineering with Shifted Rolling Windows**

**Hyperlink:** [`final_notebook.ipynb` → Section 4.4: Rolling and Season-to-Date Form Features](final_notebook.ipynb#4.4-Rolling-and-Season-to-Date-Form-Features)  
*(Also see Section 4.8: Leakage Audit for the formal verification of correctness)*

---

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**What it is:**  
We classify this under **Feature Engineering** — specifically, the engineering of temporal aggregation features that encode each team's historical performance as a pre-game signal *without* leaking information from the current game. This required engineering an interaction between the `.shift()` and `.rolling()` / `.expanding()` operations within each team-season group.

**Implementation:**  
For every basketball signal identified as predictive in the EDA (25 signals including win rate, net margin, FG%, 3P%, rebounds, assists, turnovers, etc.), we compute:

1. **Rolling window means** at three horizons (5, 10, and 20 games):
   ```python
   engineered[out_col] = ts_group[source_col].transform(
       lambda s, w=window: s.shift(1).rolling(window=w, min_periods=1).mean()
   )
   ```
   Grouping is done per `(TEAM_ID, SEASON_YEAR)` so rolling windows never cross season boundaries.

2. **Season-to-date expanding means** (9 features):
   ```python
   engineered[out_col] = ts_group[source_col].transform(
       lambda s: s.shift(1).expanding(min_periods=1).mean()
   )
   ```

3. **Matchup difference features** (21 features): each team's rolling feature minus the opponent's identical rolling feature, collapsing correlated pair-features into a single relative-edge signal.

4. **Cross-strength features** (4 features): team rolling offense vs. opponent rolling defense and vice versa.

This produced **75 rolling features + 9 season-to-date features** from 25 raw basketball signals.

**The `.shift(1)` is the critical step:** it shifts the series one game forward before the window is applied, ensuring the current game's result is never included in its own predictor. An early version of the pipeline omitted this step — training accuracy was inflated by ~10 percentage points, a clear sign of data leakage. The formal Leakage Audit (Section 4.8) confirmed the fix with five assertion checks: no raw box-score column in the feature table, no duplicate rows, exactly two rows per game, balanced target, zero NaN values.

**Season-to-date features** were added after the EDA showed that net rating correlates with season win rate at r ≈ 0.97 (Section 3.5) — a signal that accumulates over the full season rather than just the last 5–10 games.

---

### Results & Conclusion

The feature engineering pipeline produced a model-ready dataset of ~56,000 team-game rows with zero missing values and all five leakage checks passed.

The most substantive result is that **season-to-date features outperform rolling-window features** in predictive importance. Both tree models (Random Forest and XGBoost) agree that `team_minus_opp_net_rating_season_to_date` is the single strongest pre-game predictor (XGB importance: 0.2073), ranking above all rolling-window features. This directly contradicts the 'momentum' narrative in sports media and is highlighted in the Section 5 Summary: *'cumulative within-season quality is a stronger pre-game signal than short-term recent form.'*

The leakage audit validates correctness: all five structural checks passed, confirming that the `.shift(1)` engineering decision was both necessary and sufficient to produce a leakage-free feature set.


---

## Topic 2: **Hyperparameter Tuning — RandomizedSearchCV with TimeSeriesSplit**

**Hyperlink:** [`final_notebook.ipynb` → Section 5.9: Hyperparameter Tuning — Difficulty Concept](final_notebook.ipynb#5.9-Hyperparameter-Tuning-—-Difficulty-Concept)

---

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**What it is:**  
We use **Randomized Search hyperparameter tuning** — a smarter-than-grid-search method — combined with a time-aware cross-validation strategy that respects the temporal ordering of NBA games.

**Implementation:**

```python
param_distributions = {
    'n_estimators'    : [100, 200, 300, 400],
    'max_depth'       : [4, 5, 6, 7],
    'learning_rate'   : [0.03, 0.05, 0.08, 0.1],
    'subsample'       : [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma'           : [0, 0.1, 0.2],
}

tscv = TimeSeriesSplit(n_splits=3)
rs = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=tscv,
    random_state=42,
)
rs.fit(X_train_sorted, y_train_sorted)  # training data only — test never seen
```

Key design decisions:
- **`RandomizedSearchCV` with `n_iter=20`**: samples 20 random configurations from the joint distribution over 7 parameters. A full grid search over this space would require 4×4×4×3×3×3×3 = 5,184 configurations × 3 folds = 15,552 fits. Random search finds near-optimal settings in 60 fits (20 configs × 3 folds) — roughly 250× cheaper.
- **`TimeSeriesSplit(n_splits=3)`**: splits the *training set only* into 3 chronologically ordered folds. Standard k-fold would allow a 2015 validation fold to score a model trained partly on 2018 games — temporal leakage within the training window. TimeSeriesSplit ensures every validation fold comes strictly after its training fold in time.
- **`scoring='roc_auc'`**: measures ranking quality across all classification thresholds, more informative than raw accuracy for this problem.
- **Training data only**: `rs.fit(X_train_sorted, y_train_sorted)` — the test seasons (2020–2025) are never seen during tuning.

**Why we used it:**  
The XGBoost learning curve (Section 5.8) showed clear overfitting: training loss continued falling after the point where validation loss stopped improving. Default hyperparameters were too complex. RandomizedSearch with TimeSeriesSplit was the correct tool: it finds better regularization without introducing temporal leakage, and does so efficiently without an exhaustive grid.

---

### Results & Conclusion

**Best configuration found:**

| Parameter | Best Value | Effect |
|---|---|---|
| n_estimators | 100 | Fewer trees → less overfitting |
| max_depth | 4 | Shallower trees → more regularization |
| learning_rate | 0.03 | Slower learning → better generalization |
| min_child_weight | 5 | Higher leaf threshold → prunes noisy splits |
| gamma | 0.2 | Minimum loss-reduction required to split |
| colsample_bytree | 0.8 | Feature subsampling per tree |
| subsample | 0.7 | Row subsampling per tree |

- **Best CV AUC-ROC:** 0.7312
- **Improvement over default XGBoost:** +0.0032 AUC-ROC, +0.0022 accuracy
- **Tuned test performance:** 0.6946 AUC, 0.6441 accuracy

The tuner consistently moved toward more conservative settings (shallower trees, lower learning rate, higher regularization), directly addressing the overfitting shown in the learning curves — the expected and correct outcome.

The Section 5 Summary reflects these results: the CV AUC (0.7312) vs. test AUC (0.6946) gap is noted as temporal drift (the CV folds are within the 2001–2020 training era, which is easier to predict than the 2020–2025 test era), and the stakeholder conclusion explicitly states that a production model should be retrained on recent seasons to close this gap.


---

## Topic 3: **Feature Importance — Tree-Based Feature Importance Analysis (XGBoost vs. Random Forest)**

**Hyperlink:** [`final_notebook.ipynb` → Section 5.11: Feature Importance — Difficulty Concept](final_notebook.ipynb#5.11-Feature-Importance-—-Difficulty-Concept)

---

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**What it is:**  
We use **tree-based feature importance** from two independent ensemble models — tuned XGBoost and Random Forest — to identify which pre-game basketball signals most strongly drive win/loss predictions.

**Implementation:**

```python
importance_df = pd.DataFrame({
    'Feature'        : MODEL_FEATURES,
    'XGBoost (Tuned)': best_xgb.feature_importances_,
    'Random Forest'  : rf.feature_importances_,
})
```

Both models compute importance via **impurity reduction**: the aggregate reduction in split criterion (Gini for Random Forest, gradient/loss for XGBoost) summed across every node where a feature is used, normalized to sum to 1.0. Results are visualized as side-by-side horizontal bar charts (top 15 features from each model) and a paired top-10 comparison table.

We also cross-reference tree importances with the Logistic Regression coefficient plot (Section 5.6). LR coefficients identify feature *directionality* (positive vs. negative effect on win probability); tree importances identify *split value* (which features the model relies on most). Agreement across all three models provides the strongest possible evidence.

**Why two tree models instead of one?**  
Random Forest uses bagging (parallel, independent trees; Gini impurity); XGBoost uses boosting (sequential, residual-correcting trees; gradient-based). If both independently agree on the top features, that agreement is unlikely to be an artifact of either algorithm's inductive bias — it indicates genuine signal in the features themselves. Single-model importance rankings can be unstable; cross-model convergence is robust.

**Why we used feature importance at all:**  
The project goal is not just prediction accuracy but also understanding *why* teams win. Feature importance directly answers: which pre-game basketball signals actually drive the predictions? This is the interpretable complement to the accuracy numbers and directly useful to stakeholders (front offices, analysts).

---

### Results & Conclusion

Both models agree on the top-3 features:

| Rank | Feature | XGBoost Importance | RF Importance |
|---|---|---|---|
| 1 | `team_minus_opp_net_rating_season_to_date` | 0.2073 | #1 (agrees) |
| 2 | `is_home` | 0.1747 | #2 (agrees) |
| 3 | `team_minus_opp_win_pct_season_to_date` | 0.1568 | #3 (agrees) |

The top-3 features account for **~54%** of XGBoost's total importance (0.2073 + 0.1747 + 0.1568 = 0.5388).

These results are reflected in the Section 5 Summary and conclusion in three specific ways:

1. **Season-to-date quality beats short-term form.** Both cumulative features (`net_rating_season_to_date` and `win_pct_season_to_date`) rank above all rolling-window features. The conclusion calls this the most substantive basketball finding: *'cumulative within-season quality is a stronger pre-game signal than short-term recent form,'* directly contradicting the momentum narrative.

2. **Home-court advantage is quantified.** `is_home` ranks #2 in both tree models (XGB: 0.1747) and #1 in the Logistic Regression coefficients — consistent across all three models. This aligns with the EDA permutation test (Section 3.7.1) showing home teams win 58.6% of games, and the EDA trend test (Section 3.7.4) showing this effect is declining at −0.0026 per season (p < 0.001).

3. **Stakeholder implication.** The Section 5 Summary explicitly uses the feature importance findings to advise front offices: *'Cumulative season-to-date net rating differential is the strongest pre-game predictor — more reliable than short-term streaks. Front offices can use this signal to quantify matchup edges and benchmark expected win probability against actual results.'*
